# Kapitel 2 - Ett ML projekt från början till slut

## Faktafrågor

### 1. I kapitlet beskrivs en checklista med sju steg. Beskriv de sju stegen översiktligt. Följs dessa steg i en rak progression eller arbetar man mer iterativt?

De sju stegen är:
1. Kolla helheten (förstå problemet/målet)
2. Kolla upp datasetet, var det finns, och ladda in det i notebooken
3. Utforska datan (EDA)
4. Rensa/tvätta datan om det behövs
5. ML-modellering: välj olika modeller och träna dem
6. Presentera lösningen och resultatet
7. Placera lösningen i produktion så att den kan användas i verkligheten

I verkligheten följer man sällan stegen i en helt rak progression, utan arbetar mer iterativt — man går ofta fram och tillbaka mellan t.ex. EDA, datarensning och modellering allteftersom man lär sig mer om datan och problemet.

### 2. Vad menas med att en modell produktionssätts?

Att en modell produktionssätts betyder att den flyttas från labbet till verklig användning i ett system, och görs tillgänglig för användare. Om t.ex. en bilvärderingsmodell har byggts så placeras den på en webbsida där den kan nyttjas av riktiga kunder.

### 3. Vad är scikit-learn för något? Biblioteket följer några centrala designprinciper. Vilka är dessa? Vad är estimators, predictors och transformers?

Scikit-learn är klassiskt pythons mest kända och använda bibliotek för maskininlärning. Den har inbyggda algoritmer med formler och logik så vi kan implementera modeller snabbare.

Biblioteket bygger på att alla objekt följer samma konsekventa API, det gör det lätt att kombinera olika delar t ex i en Pipeline.

Estimator är objekt som kan uppskatta parametrar utifrån data via .fit(). Predictor är en estimator som också kan göra förutsägelser via .predict(), t ex LinearRegression. Transformer är en estimator som omvandlar data via .transform(), t ex StandardScaler eller SimpleImputer.

### 4. Vad är TensorFlow och Keras?

TensorFlow och Keras är kända pythonbibliotek som används främst för djupinlärning och neurala nätverk. TensorFlow klarar av att hantera gigantiska datamängder och tunga matematiska beräkningar. Keras är enklare, byggs ovanpå TensorFlow och används för att bygga modeller snabbare utan att behöva skriva så detaljerad kod.

## Resonemangfrågor

### 5. Kalle säger "om jag tränat en modell och den inte presterar bra nog på testdatan så justerar jag den tills den gör det." Stina säger "det är ett stort fel att göra så, det enda du då åstadkommer är att du överanpassar testdatan. Hela syftet med testdatan försvinner då". Vad säger du om deras dialog?

Stina har rätt. Testdatan är verkligheten, och Kalles lösning att justera modellen tills den presterar bra på testdatan gör bara att modellen blir överanpassad och oförutsägbar i sina prediktioner på ny data. Testdatan borde hållas orörd och bara användas en gång i slutet. Det är valideringsdatan som ska användas för att justera inställningar/hyperparametrar för respektive modell.

### 6. Många AI/ML projekt uppnår inte de ursprungligen satta målen eller att ens passera någon form av prototyp-stadie. Vad tror du detta beror på och hur ska vi förhålla oss till det?

Det misslyckas oftast pga dåligt indata, otydliga mål, svåra integrationer på plats i verklig drift, och för stora förväntningar.

Bättre sätt att förhålla sig till det: behålla enkelhet, jobba stegvis/iterativt, och planera för verkligheten redan från början med en översikt av systemet som modellen ska integreras med.

## Koduppgifter

### 8. Förklara vad koden nedan gör. Varför är det viktigt att kunna spara en modell?

In [1]:
# Kod att förklara (körs inte nödvändigtvis här, bara analyseras):
#
# from sklearn.datasets import make_regression
# from sklearn.linear_model import LinearRegression
# from joblib import dump, load
#
# X, y = make_regression(n_samples=20000, n_features=3, noise=0.1)
# model = LinearRegression().fit(X, y)
# dump(model, "linear_model.joblib")
# model_loaded = load("linear_model.joblib")
# print(model_loaded.predict(X[:5]))

Koden gör följande:
1. Skapar låtsasdata med 20000 rader och 3 kolumner (make_regression)
2. Tränar en LinearRegression-modell på datan
3. Sparar modellen till en fil med joblib.dump
4. Laddar in modellen igen med joblib.load
5. Gör en prediktion/gissning med den inladdade modellen

Det är viktigt att kunna spara en modell för att det sparar tid och datakraft, man slipper träna om modellen varje gång. Man kan då också använda den sparade modellen i appar och på webbsidor, alltså i produktion.

### 9. Läs in datasetet "data_01.csv", dela upp i X/y, träna/validering/test, träna två regressionsmodeller, utvärdera, träna om på bäst presterande modell.

In [2]:
import pandas as pd                                    # Läsa in och hantera data(tabeller)
from sklearn.model_selection import train_test_split   # Dela upp data i train/val/test
from sklearn.linear_model import LinearRegression      # Modell 1: rak formel
from sklearn.tree import DecisionTreeRegressor          # Modell 2: ja/nej-frågor
from sklearn.metrics import root_mean_squared_error     # Mäta hur fel modellerna gissar (RMSE)

## Läsa in data

In [3]:
df = pd.read_csv('data/data_01.csv')
df.head()

,x1,x2,x3,x4,x5,target
0,0.743487,1.072825,1.332911,-1.244771,0.344978,220.173943
1,0.835264,0.202184,0.966480,0.745883,-0.033773,175.873929
2,-1.103234,0.030615,-0.140385,0.727683,-2.831224,-162.270054
3,1.210186,1.685258,-0.394123,0.719024,-2.166585,165.930461
4,0.474577,0.647737,-0.451812,-0.409472,-0.051473,43.250511


## Dela upp i X och Y

In [4]:
X = df.drop(columns=['target'])
y = df['target']

## Train, Validation, Test

In [5]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.1875, random_state=42)

print(X_train.shape, X_val.shape, X_test.shape)

(129, 5) (30, 5) (40, 5)


## Träna två modeller

In [6]:
lin_reg = LinearRegression()             # skapar modell 1, linjär regression
lin_reg.fit(X_train, y_train)            # tränar modellen på träningsdatan
lin_reg_pred = lin_reg.predict(X_val)    # gissar på valideringsdatan

tree_reg = DecisionTreeRegressor(random_state=42)   # skapar modell 2, beslutsträd
tree_reg.fit(X_train, y_train)                      # tränar den också på träningsdatan
tree_reg_pred = tree_reg.predict(X_val)             # gissar på samma valideringsdata

## Utvärdera på valideringsdatan

In [7]:
lin_reg_rmse = root_mean_squared_error(y_val, lin_reg_pred)    # räknar ut felet för lin reg
tree_reg_rmse = root_mean_squared_error(y_val, tree_reg_pred)  # räknar ut felet för trädet

print("RMSE Linear Regression:", lin_reg_rmse)   # skriver ut resultat lin reg
print("RMSE Decision Tree:", tree_reg_rmse)       # skriver ut resultat träd, lägre = bättre

RMSE Linear Regression: 3.3240790627529435
RMSE Decision Tree: 96.01575547055431


## Träna om bästa modellen (lin reg) på train+val, testa på testsetet

In [8]:
lin_reg_final = LinearRegression()                          # ny modell för slutgiltig träning
lin_reg_final.fit(X_train_full, y_train_full)               # tränar på train+val ihop
test_pred = lin_reg_final.predict(X_test)                   # gissar på osedd testdata

test_rmse = root_mean_squared_error(y_test, test_pred)      # räknar ut felet på testdatan
print("RMSE på testdata:", test_rmse)                        # ärligt slutresultat

RMSE på testdata: 3.371549077515551


## Träna om modellen på hela datasetet

In [9]:
final_model = LinearRegression()   # sista modellen, redo för produktion
final_model.fit(X, y)              # tränar på ALL data (train+val+test ihop)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](5,)","[77.24,87.85,71.77,28.45,31.66]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](5,)","['x1','x2','x3','x4','x5']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,0.1269
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,5
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(5)


### 10. Datasetet "salary_dataset.csv": läs in, dela i X/y, träning/test-split (inget valideringsset), träna två regressionsmodeller med k-delad korsvalidering (cross_validate, neg_root_mean_squared_error), utvärdera bästa modellen på testsetet.

In [10]:
df_salary = pd.read_csv('data/salary_dataset.csv')   # läser in datasetet
df_salary.head()                                       # kollar hur det ser ut

,YearsExperience,Salary
0,1.2,39344.0
1,1.4,46206.0
2,1.6,37732.0
3,2.1,43526.0
4,2.3,39892.0


In [11]:
X_salary = df_salary[['YearsExperience']]   # X måste vara 2D, därför dubbla hakparenteser
y_salary = df_salary['Salary']              # y är det vi vill förutsäga

In [12]:
X_salary_train, X_salary_test, y_salary_train, y_salary_test = train_test_split(
    X_salary, y_salary, test_size=0.2, random_state=42
)   # bara train/test denna gång, ingen val eftersom vi kör korsvalidering istället

In [13]:
from sklearn.model_selection import cross_validate   # för k-delad korsvalidering

lin_reg_salary = LinearRegression()          # modell 1
tree_reg_salary = DecisionTreeRegressor(random_state=42)   # modell 2

lin_scores = cross_validate(lin_reg_salary, X_salary_train, y_salary_train,
                             cv=5, scoring='neg_root_mean_squared_error')   # 5-delad cv, lin reg

tree_scores = cross_validate(tree_reg_salary, X_salary_train, y_salary_train,
                              cv=5, scoring='neg_root_mean_squared_error')  # 5-delad cv, träd

In [14]:
import numpy as np   # för medelvärde

lin_rmse_cv = -np.mean(lin_scores['test_score'])     # gör om till positivt, tar medelvärde
tree_rmse_cv = -np.mean(tree_scores['test_score'])   # samma för träd

print("CV RMSE Linear Regression:", lin_rmse_cv)
print("CV RMSE Decision Tree:", tree_rmse_cv)

CV RMSE Linear Regression: 5293.203196998775
CV RMSE Decision Tree: 5611.57858424216


In [15]:
lin_reg_salary.fit(X_salary_train, y_salary_train)         # tränar bästa modellen på all träningsdata
salary_test_pred = lin_reg_salary.predict(X_salary_test)   # gissar på testdatan

salary_test_rmse = root_mean_squared_error(y_salary_test, salary_test_pred)   # räknar felet
print("RMSE på testdata:", salary_test_rmse)

RMSE på testdata: 7059.043621901506


In [16]:
salary_test_rmse / np.mean(y_salary_test)   # felet i % av medellönen, lägre = bättre

np.float64(0.08485906344136306)

### 11. Arbeta med kategorisk data: datasetet "mpg" (seaborn) — droppa saknade värden, droppa kolumnen "name", dummy-variable-encoding på "origin" (drop_first=True), dela i X/y (mpg = y), träning/test-split, träna en linjär regressionsmodell och utvärdera på testdatan.

In [17]:
import seaborn as sns

df_mpg = sns.load_dataset("mpg")   # läser in mpg-datasetet från seaborn
df_mpg.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


In [18]:
df_mpg = df_mpg.dropna()              # tar bort rader med saknade värden
df_mpg = df_mpg.drop(columns=['name'])   # namnet på bilen är inte relevant för mpg

In [19]:
df_mpg = pd.get_dummies(df_mpg, columns=['origin'], drop_first=True)   # gör om origin till 2 nya 0/1-kolumner
df_mpg.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin_japan,origin_usa
0,18.0,8,307.0,130.0,3504,12.0,70,False,True
1,15.0,8,350.0,165.0,3693,11.5,70,False,True
2,18.0,8,318.0,150.0,3436,11.0,70,False,True
3,16.0,8,304.0,150.0,3433,12.0,70,False,True
4,17.0,8,302.0,140.0,3449,10.5,70,False,True


In [20]:
X_mpg = df_mpg.drop(columns=['mpg'])   # allt utom mpg blir X
y_mpg = df_mpg['mpg']                   # mpg är det vi vill förutsäga

X_mpg_train, X_mpg_test, y_mpg_train, y_mpg_test = train_test_split(
    X_mpg, y_mpg, test_size=0.2, random_state=42
)

In [21]:
lin_reg_mpg = LinearRegression()
lin_reg_mpg.fit(X_mpg_train, y_mpg_train)             # tränar modellen
mpg_test_pred = lin_reg_mpg.predict(X_mpg_test)       # gissar på testdatan

mpg_test_rmse = root_mean_squared_error(y_mpg_test, mpg_test_pred)   # räknar felet
print("RMSE på testdata:", mpg_test_rmse)

RMSE på testdata: 3.256114096847396


### 12. I avsnitt 2.2 "Huspriser i Kalifornien" fick vi RMSE Random Forest Regression: 52277.96578719621 på valideringsdatan. Försök få ett bättre resultat.

Se `kapitel2_huspriser_exempel.ipynb` — där finns hela POC:en (EDA, imputering, train/val/test) och en Random Forest-lösning som försöker slå 52277.97.